# Multi-Model Serving: 10 SKLearn Models on a Single Serving Endpoint

This notebook demonstrates **Databricks Multi-Model Serving** by deploying **10 copies** of the same SKLearn model to a **single Model Serving endpoint** with custom traffic routing.

**Traffic Split Strategy:**
- **Model 1 ("Hot" Model):** Receives **50%** of all inference traffic
- **Models 2–10 ("Cold" Models):** Split the remaining **50%** as evenly as possible (~5–6% each)

**Why?** This pattern is useful for A/B testing, canary deployments, and validating that multi-model endpoints can efficiently route traffic across many served entities.

**Reference:** [Serve multiple models to a serving endpoint](https://docs.databricks.com/aws/en/machine-learning/model-serving/serve-multiple-models-to-serving-endpoint)

In [0]:
# ============================================================
# User-configurable parameters — update these for your environment
# ============================================================

CATALOG = "ram"
SCHEMA = "multi_model_serving"
MODEL_BASE_NAME = "sklearn_mme_model"   # base name; models will be named {base}_1 through {base}_10
ENDPOINT_NAME = "mme-sklearn-poc"
NUM_MODELS = 10
HOT_MODEL_INDEX = 1    # which model gets the majority of traffic
HOT_TRAFFIC_PCT = 50   # percentage of traffic for the hot model

# Ensure the catalog and schema exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Using: {CATALOG}.{SCHEMA}")

In [0]:
%pip install scikit-learn mlflow databricks-sdk --upgrade --quiet
dbutils.library.restartPython()

In [0]:
# Re-import after Python restart
import mlflow
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

# Point MLflow to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# Re-define config variables (Python state was cleared by restartPython)
CATALOG = "ram"
SCHEMA = "multi_model_serving"
MODEL_BASE_NAME = "sklearn_mme_model"
ENDPOINT_NAME = "mme-sklearn-poc"
NUM_MODELS = 10
HOT_MODEL_INDEX = 1
HOT_TRAFFIC_PCT = 50

In [0]:
# ============================================================
# Generate synthetic regression data and train a RandomForest
# ============================================================

# Synthetic dataset: 1000 samples, 10 features
X, y = make_regression(n_samples=1000, n_features=10, noise=0.1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

score = model.score(X_test, y_test)
print(f"RandomForestRegressor R² on test set: {score:.4f}")
print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}, Features: {X_train.shape[1]}")

## Register 10 Model Versions in Unity Catalog

We register **10 separate models** (not 10 versions of the same model) in Unity Catalog. Each model is a distinct registered entity so it can be served as its own `ServedEntity` on the multi-model endpoint.

Naming convention: `{CATALOG}.{SCHEMA}.{MODEL_BASE_NAME}_{i}` for i = 1..10

In [0]:
# ============================================================
# Register 10 copies of the trained model in Unity Catalog
# ============================================================

model_registry_info = []  # store model info for endpoint creation

for i in range(1, NUM_MODELS + 1):
    model_name = f"{CATALOG}.{SCHEMA}.{MODEL_BASE_NAME}_{i}"
    served_entity_name = f"sklearn-model-{i}"

    with mlflow.start_run(run_name=f"register_{MODEL_BASE_NAME}_{i}") as run:
        # Log the trained model and register it in Unity Catalog
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            registered_model_name=model_name,
            input_example=X_test[:1],  # single-row input example for signature inference
        )

    # Get the latest version (should be 1 for fresh models)
    client = mlflow.MlflowClient()
    latest_versions = client.search_model_versions(f"name='{model_name}'")
    version = max(int(v.version) for v in latest_versions)

    model_registry_info.append({
        "model_name": model_name,
        "version": str(version),
        "served_entity_name": served_entity_name,
    })

    print(f"  Registered: {model_name} (version {version}) → served as '{served_entity_name}'")

print(f"\nTotal models registered: {len(model_registry_info)}")

## Create Multi-Model Serving Endpoint

**Traffic routing strategy:**
| Model | Role | Traffic % |
|-------|------|-----------|
| sklearn-model-1 | Hot | 50% |
| sklearn-model-2 through sklearn-model-10 | Cold | ~5–6% each |

Since 50 ÷ 9 = 5 remainder 5, the first 5 cold models get **6%** and the last 4 cold models get **5%**, totaling exactly **100%**.

In [0]:
import mlflow.deployments

client = mlflow.deployments.get_deploy_client("databricks")

# ============================================================
# Build served entities list
# ============================================================
served_entities = []
for info in model_registry_info:
    served_entities.append({
        "name": info["served_entity_name"],
        "entity_name": info["model_name"],
        "entity_version": info["version"],
        "workload_size": "Small",
        "workload_type": "CPU",
        "scale_to_zero_enabled": True,
    })

# ============================================================
# Build traffic config: hot model gets HOT_TRAFFIC_PCT,
# remaining traffic split evenly among cold models
# ============================================================
remaining_pct = 100 - HOT_TRAFFIC_PCT  # 50
num_cold = NUM_MODELS - 1               # 9
base_pct = remaining_pct // num_cold     # 5
extra = remaining_pct % num_cold         # 5 (first 5 cold models get 6%)

routes = []
for info in model_registry_info:
    i = int(info["served_entity_name"].split("-")[-1])  # extract model index
    if i == HOT_MODEL_INDEX:
        pct = HOT_TRAFFIC_PCT
    else:
        # Cold models: first `extra` cold models get base_pct + 1
        cold_rank = i - 1 if i > HOT_MODEL_INDEX else i  # rank among cold models
        pct = base_pct + 1 if cold_rank <= extra else base_pct

    routes.append({
        "served_model_name": info["served_entity_name"],
        "traffic_percentage": pct,
    })

# Sanity check
total_traffic = sum(r["traffic_percentage"] for r in routes)
assert total_traffic == 100, f"Traffic must sum to 100%, got {total_traffic}%"

# ============================================================
# Create the endpoint (skip if it already exists)
# ============================================================
print(f"Creating endpoint: {ENDPOINT_NAME}")
print(f"Served entities: {len(served_entities)}")
print(f"\nTraffic split:")
for r in routes:
    role = "HOT" if r["traffic_percentage"] == HOT_TRAFFIC_PCT else "cold"
    print(f"  {r['served_model_name']:20s} \u2192 {r['traffic_percentage']:3d}%  ({role})")

try:
    endpoint = client.create_endpoint(
        name=ENDPOINT_NAME,
        config={
            "served_entities": served_entities,
            "traffic_config": {"routes": routes},
        },
    )
    print(f"\nEndpoint '{ENDPOINT_NAME}' creation initiated.")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"\nEndpoint '{ENDPOINT_NAME}' already exists \u2014 skipping creation.")
        print("To update traffic config, delete and recreate or use the update API.")
    else:
        raise e

## Check Endpoint Status

Poll the endpoint until all served models are **READY**. This can take 10–20 minutes on initial deployment.

In [0]:
import time

# ============================================================
# Poll the endpoint until it reaches READY state
# ============================================================
MAX_WAIT_SECONDS = 30 * 60   # 30 minutes
POLL_INTERVAL = 60            # check every 60 seconds

start_time = time.time()
print(f"Waiting for endpoint '{ENDPOINT_NAME}' to become READY...\n")

while True:
    ep = client.get_endpoint(endpoint=ENDPOINT_NAME)
    state = ep.get("state", {})
    ready = state.get("ready", "UNKNOWN")
    config_update = state.get("config_update", "UNKNOWN")
    elapsed = int(time.time() - start_time)

    print(f"[{elapsed:>4d}s] Ready: {ready}  |  Config update: {config_update}")

    if ready == "READY":
        print(f"\nEndpoint '{ENDPOINT_NAME}' is READY! (took {elapsed}s)")
        break

    if elapsed >= MAX_WAIT_SECONDS:
        print(f"\nTimed out after {MAX_WAIT_SECONDS}s. Check the endpoint in the UI.")
        break

    time.sleep(POLL_INTERVAL)

## Test Inference

Send sample data to the endpoint. By default, traffic is routed according to the configured percentages. You can also target a **specific served model** directly.

In [0]:
import mlflow.deployments
import requests
import json

client = mlflow.deployments.get_deploy_client("databricks")

# ============================================================
# Build sample input matching the 10 features from training
# ============================================================
columns = [f"feature_{j}" for j in range(X_test.shape[1])]
sample_data = [float(v) for v in X_test[0]]

payload = {
    "dataframe_split": {
        "index": [0],
        "columns": columns,
        "data": [sample_data],
    }
}

print("Sample payload:")
print(json.dumps(payload, indent=2))

# ============================================================
# 1. Traffic-routed query (endpoint decides which model serves
#    based on the configured traffic percentages)
# ============================================================
host = mlflow.utils.databricks_utils.get_workspace_url().rstrip("/")
traffic_url = f"{host}/serving-endpoints/{ENDPOINT_NAME}/invocations"

print("\n" + "=" * 60)
print("TRAFFIC-ROUTED QUERY (mlflow.deployments)")
print("=" * 60)
print(f"  Endpoint:  {ENDPOINT_NAME}")
print(f"  URL:       {traffic_url}")
print(f"  Routing:   Automatic — model selected by traffic split")
response = client.predict(endpoint=ENDPOINT_NAME, inputs=payload)
print(f"  Response:  {response}")

# ============================================================
# 2. Query a SPECIFIC served model directly
#    The mlflow.deployments client does not support targeting
#    individual served models. Use the REST API instead:
#      POST /serving-endpoints/{endpoint}/served-models/{model}/invocations
# ============================================================
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

def query_served_model(model_name):
    """Query a specific served model by name, bypassing traffic routing."""
    url = f"{host}/serving-endpoints/{ENDPOINT_NAME}/served-models/{model_name}/invocations"
    print(f"\n{'=' * 60}")
    print(f"DIRECT QUERY: {model_name}")
    print(f"{'=' * 60}")
    print(f"  Model:     {model_name}")
    print(f"  URL:       {url}")
    print(f"  Routing:   Direct — bypasses traffic split")
    resp = requests.post(url, headers=headers, json=payload)
    resp.raise_for_status()
    result = resp.json()
    print(f"  Response:  {result}")
    return result

result_hot = query_served_model("sklearn-model-1")
result_cold = query_served_model("sklearn-model-4")

# ============================================================
# To query ANY of the 10 models directly, call:
#   query_served_model("sklearn-model-{i}")  where i = 1..10
# ============================================================

## Cleanup

Uncomment the cell below to delete the serving endpoint and all registered models.

In [0]:
# ============================================================
# CLEANUP — Uncomment to delete endpoint and registered models
# ============================================================

# # 1. Delete the serving endpoint
# print(f"Deleting endpoint: {ENDPOINT_NAME}")
# client.delete_endpoint(endpoint=ENDPOINT_NAME)
# print("Endpoint deleted.")

# # 2. Delete registered models from Unity Catalog
# from mlflow import MlflowClient
# ml_client = MlflowClient()
# for i in range(1, NUM_MODELS + 1):
#     model_name = f"{CATALOG}.{SCHEMA}.{MODEL_BASE_NAME}_{i}"
#     try:
#         versions = ml_client.search_model_versions(f"name='{model_name}'")
#         for v in versions:
#             ml_client.delete_model_version(name=model_name, version=v.version)
#         ml_client.delete_registered_model(name=model_name)
#         print(f"  Deleted: {model_name}")
#     except Exception as e:
#         print(f"  Could not delete {model_name}: {e}")

# print("\nCleanup complete.")